In [1]:
from heapq import heappush, heappop

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)   # 0 = blank

GOAL_POS = {v: i for i, v in enumerate(GOAL)}  # tile -> index in GOAL

# Precomputed legal blank moves by index (0..8) on a 3x3 grid
MOVES = {
    0: (1, 3),
    1: (0, 2, 4),
    2: (1, 5),
    3: (0, 4, 6),
    4: (1, 3, 5, 7),
    5: (2, 4, 8),
    6: (3, 7),
    7: (4, 6, 8),
    8: (5, 7),
}

def h1_misplaced(state):
    """H1: number of misplaced tiles (ignoring blank)."""
    return sum(1 for i, v in enumerate(state) if v != 0 and v != GOAL[i])

def h2_manhattan(state):
    """H2: sum of Manhattan distances of tiles from goal positions (ignoring blank)."""
    dist = 0
    for i, v in enumerate(state):
        if v == 0:
            continue
        gi = GOAL_POS[v]
        dist += abs(i // 3 - gi // 3) + abs(i % 3 - gi % 3)
    return dist

def neighbors(state):
    """Generate neighboring states by sliding a tile into the blank."""
    z = state.index(0)
    for nz in MOVES[z]:
        s = list(state)
        s[z], s[nz] = s[nz], s[z]
        yield tuple(s)

def is_solvable(state):
    """
    For 3x3, puzzle is solvable iff inversion count is even (ignoring blank).
    """
    arr = [x for x in state if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0

def reconstruct_path(parent, goal_state):
    path = []
    cur = goal_state
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()
    return path

def astar(start, heuristic):
    """
    A* search. Returns:
      - path (list of states)
      - depth (moves)
      - nodes_explored (expanded states)
    """
    if not is_solvable(start):
        return None

    # g-cost map and parent pointers
    g_cost = {start: 0}
    parent = {start: None}

    # priority queue items: (f, g, tie_breaker, state)
    open_heap = []
    tie = 0
    heappush(open_heap, (heuristic(start), 0, tie, start))
    tie += 1

    closed = set()
    nodes_explored = 0

    while open_heap:
        f, g, _, state = heappop(open_heap)
        if state in closed:
            continue

        closed.add(state)
        nodes_explored += 1

        if state == GOAL:
            path = reconstruct_path(parent, state)
            return {
                "path": path,
                "depth": len(path) - 1,
                "nodes_explored": nodes_explored
            }

        for nb in neighbors(state):
            if nb in closed:
                continue
            new_g = g + 1
            if new_g < g_cost.get(nb, float("inf")):
                g_cost[nb] = new_g
                parent[nb] = state
                heappush(open_heap, (new_g + heuristic(nb), new_g, tie, nb))
                tie += 1

    return None

def print_state(s):
    for r in range(0, 9, 3):
        row = s[r:r+3]
        print(" ".join("_" if x == 0 else str(x) for x in row))
    print()

def compare(start):
    print("Start state:")
    print_state(start)

    for name, h in [("H1 Misplaced Tiles", h1_misplaced),
                    ("H2 Manhattan Distance", h2_manhattan)]:
        result = astar(start, h)
        if result is None:
            print(f"{name}: Unsolvable start state.\n")
            continue

        print(f"{name}:")
        print(f"  Solution depth:   {result['depth']}")
        print(f"  Nodes explored:   {result['nodes_explored']}")
        print()

if __name__ == "__main__":
    # Example harder test case (known to require 20 moves to solve to the standard goal)
    start_state = (7, 2, 4,
                   5, 0, 6,
                   8, 3, 1)
    compare(start_state)


Start state:
7 2 4
5 _ 6
8 3 1

H1 Misplaced Tiles:
  Solution depth:   20
  Nodes explored:   3667

H2 Manhattan Distance:
  Solution depth:   20
  Nodes explored:   283

